# `Reporting Excel automatisé en Python`
*Oscar JOSEPH--GENESLAY*
**Data :** SuperStore Dataset
Source : *[Kaggle - SuperStore Dataset par vivek468](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final?resource=download)*

# Importation des packages

In [389]:
import pandas as pd
from datetime import datetime
import openpyxl
from openpyxl.chart import BarChart, Reference, DoughnutChart, series
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.utils import FORMULAE
from openpyxl.chart.series import DataPoint
from openpyxl.chart.label import DataLabelList
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.worksheet.datavalidation import DataValidation
from pathlib import Path
import pandas as pd
import os
import boto3

# Importation des données

In [390]:
dataset = pd.read_csv(
    "https://minio.lab.sspcloud.fr/oscar04/Superstore/Superstore.csv",
    encoding="windows-1252"
)

dataset.head(3)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


In [391]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

# Reformatage des colonnes

In [392]:
# Retypage
dataset["Row ID"] = str(dataset["Row ID"])
dataset["Postal Code"] = str(dataset["Postal Code"])


# Transformation en format date
dataset["Order Date"] = pd.to_datetime(dataset["Order Date"], format='%d/%m/%Y', errors='coerce')
dataset["Ship Date"] = pd.to_datetime(dataset["Ship Date"], format='%d/%m/%Y', errors='coerce')

dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   str           
 1   Order ID       9994 non-null   str           
 2   Order Date     4042 non-null   datetime64[us]
 3   Ship Date      3898 non-null   datetime64[us]
 4   Ship Mode      9994 non-null   str           
 5   Customer ID    9994 non-null   str           
 6   Customer Name  9994 non-null   str           
 7   Segment        9994 non-null   str           
 8   Country        9994 non-null   str           
 9   City           9994 non-null   str           
 10  State          9994 non-null   str           
 11  Postal Code    9994 non-null   str           
 12  Region         9994 non-null   str           
 13  Product ID     9994 non-null   str           
 14  Category       9994 non-null   str           
 15  Sub-Category   9994 non-null   s

# Fichier du reporting

## Création du fichier Excel

In [393]:
# Créer le chemin d'export
chemin_dossier = Path("../output")
chemin_dossier.mkdir(parents=True, exist_ok=True)

# On exporte le fichier de travail
path = "../output/reporting.xlsx"
with pd.ExcelWriter(path, engine="openpyxl") as writer:
    dataset.to_excel(writer, sheet_name="DATA", index=False)

print("✅ reporting.xlsx créé")

✅ reporting.xlsx créé


## Chargement du reporting en mémoire

In [394]:
# Charger le classeur (workbook) en mémoire
wb = openpyxl.load_workbook('../output/reporting.xlsx')

# Création d'une feuille visualisations
ws_visualisations = wb.create_sheet(title="VISUALISATIONS")
ws_visualisations.sheet_view.showGridLines = False

# Définir la feuille DATA en variable python
ws_data = wb["DATA"]

# wb est un objet Python — on ne peut pas afficher son contenu avec un simple print
print(type(wb))
print('Feuilles disponibles :', wb.sheetnames)

<class 'openpyxl.workbook.workbook.Workbook'>
Feuilles disponibles : ['DATA', 'VISUALISATIONS']


# Création des KPIs

### KPIs numériques

##### CA ; Profits ; Quantité

In [395]:
# Création fonction d'automatisation
def numerical_kpis(cell, colonne, cell_format):
    c = ws_visualisations[cell]
    c.value = f"=SUMIFS(DATA!{colonne}:{colonne}, DATA!M:M, VISUALISATIONS!B5, DATA!K:K, VISUALISATIONS!B11)"
    c.number_format = cell_format
    c.font = Font(name="Calibri", size=16, bold=True, color="1F497D")
    c.alignment = Alignment(horizontal="left", vertical="center")

    # Ajustement automatique de la largeur pour éviter les ####
    col_letter = c.column_letter
    # On récupère la largeur actuelle ou une valeur par défaut de 10
    current_width = ws_visualisations.column_dimensions[col_letter].width or 10
    # définit une largeur de marge
    needed_width = max(len(cell_format) + 5, 18)
    # applique la largeur
    ws_visualisations.column_dimensions[col_letter].width = max(current_width, needed_width)
    ws_visualisations.row_dimensions[c.row].height = 30

In [396]:
# CA
numerical_kpis("C4", "R", "#,##0.0 $")

# Profit
numerical_kpis("E4", "U", "#,##0.0 $")

# Quantité
numerical_kpis("G4", "S", "#,##0")

##### Temps de livraison

La fonction de calcul de différence entre date existe dans openpyxl

In [397]:
"DATEDIF" in FORMULAE

True

Application de la formule datedif sur chaque ligne où c'est calculable

In [398]:
for row_cell in range(2, len(dataset)+2):
    cell_application = f"V{row_cell}"
    date_debut = f"C{row_cell}"
    date_fin = f"D{row_cell}"
    ws_data[cell_application] = f'=IF(OR({date_debut} = "", {date_fin} = ""), "", DATEDIF({date_debut}, {date_fin}, "D"))'

Création du KPI moyenne 

In [399]:
"AVERAGEIF" in FORMULAE

True

In [400]:
ws_visualisations["J4"] = "=ROUND(AVERAGEIFS(DATA!V:V, DATA!M:M, VISUALISATIONS!B5, DATA!K:K, VISUALISATIONS!B11),1)"

In [401]:
c = ws_visualisations["J4"]
cell_format = c.number_format or ""
c.font = Font(name="Calibri", size=16, bold=True, color="1F497D")
c.alignment = Alignment(horizontal="left", vertical="center")

# Ajustement de la hauteur de la ligne 4
ws_visualisations.row_dimensions[4].height = 30
# ajustement de la largeur de la colonne O
col_letter = "O"
current_width = ws_visualisations.column_dimensions[col_letter].width or 10
# Définit une largeur de marge basée sur le format
needed_width = max(len(cell_format) + 5, 18)
# Appliquer la largeur finale
ws_visualisations.column_dimensions[col_letter].width = max(current_width, needed_width)

### TOP10 des sous-catégories avec le plus de profits

1. Calcul des profits par sous catégories et création du top 10

La fonction UNIQUE n'existe pas dans openpyxl

In [402]:
"UNIQUE" in FORMULAE

False

Calcul du nombre de sous catégories

In [403]:
len_sous_cat = len(dataset["Sub-Category"].unique())+1
# Calcul du nombre de sous catégories
len_dict ={}
for col in ["Sub-Category"]:
    len_dict["len_sous_cat"] = len(dataset[col].unique())+1 # on rajoute +1 car python commence à 0
len_dict

{'len_sous_cat': 18}

Application de la formule UNIQUE

In [404]:
formula = "=_xlfn.UNIQUE(DATA!P:P)"
ws_data['Z2'] = ArrayFormula(f"Z2:Z{len_dict['len_sous_cat']+1}", formula)

Calculer le profits par sous catégories

In [405]:
for num_cell in range(3, len_dict['len_sous_cat']+2):
    cell = "AA" + str(num_cell)
    ws_data[cell] = "=SUMIFS(DATA!U:U,DATA!P:P,DATA!Z" + str(num_cell) + ",DATA!M:M,Visualisations!$B$5, DATA!K:K, VISUALISATIONS!$B$11)"

La fonction "SORT" n'existe pas dans openpyxl

In [406]:
"SORT" in FORMULAE

False

Application du tri du tableau de données

In [407]:
# Trier le tableau de données
formula = "=_xlfn.SORT(DATA!Z3:AA19,2,-1)"
ws_data['AC3'] = ArrayFormula("AC3:AD19", formula)

2. Création du graphique à barres

In [408]:
# Création du graphique à barres
chart_top10 = BarChart()
chart_top10.type = "col"
chart_top10.title = "TOP 10 du profit en fonction des sous-catégories"
chart_top10.width = 18
chart_top10.height = 10

# Récupération des données de profits
data_ref = Reference(ws_data, min_col=30, min_row=3, max_row=12)

# Récupération des labels
cats_ref = Reference(ws_data, min_col=29, min_row=4, max_row=12)

# Appliquer les paramètres au graphique
chart_top10.add_data(data_ref, titles_from_data=True)
chart_top10.set_categories(cats_ref)

# Ajouter les étiquettes de données
chart_top10.dataLabels = DataLabelList()
chart_top10.dataLabels.showVal = True
chart_top10.dataLabels.numFmt = '#,##0'

# Titres des axes
chart_top10.y_axis.title = 'Profits (en $)'
chart_top10.x_axis.title = 'Sous-catégories'

# Légende
chart_top10.legend = None

# Placement du graphique
ws_visualisations.add_chart(chart_top10, "C7")

### Pourcentage de CA par catégories

Écrire les valeurs uniques des catégories disponibles

In [409]:
len_cat = len(dataset["Category"].unique())+1
# Calcul du nombre de sous catégories
len_dict ={}
for col in ["Category"]:
    len_dict["len_cat"] = len(dataset[col].unique())+1 # on rajoute +1 car python commence à 0
formula = "=_xlfn.UNIQUE(DATA!O:O)"
ws_data['Z25'] = ArrayFormula(f"Z25:Z{len_dict['len_cat']+24}", formula)

Calculer le CA pour chaque catégorie

In [410]:
for row_cell in range(26, 29):
    ws_data[f"AA{row_cell}"] = f"=SUMIFS(DATA!R:R, DATA!O:O, DATA!Z{row_cell}, DATA!M:M, Visualisations!$B$5, DATA!K:K, VISUALISATIONS!$B$11)"

Calculer le pourcentage du CA par catégorie

In [411]:
ws_data["AA29"] = "=SUM(AA26:AA28)"
for row_cell in range(26, 29):
    ws_data[f"AB{row_cell}"] = f"=AA{row_cell} / AA29"
    ws_data[f"AB{row_cell}"].number_format = '0.00%'

Génération du graphique

In [412]:
chart_percent = DoughnutChart()
chart_percent.width = 18
chart_percent.height = 10
labels = Reference(ws_data, min_col=26, min_row=26, max_row=28)
data = Reference(ws_data, min_col=28, min_row=25, max_row=28)
chart_percent.add_data(data, titles_from_data=True)
chart_percent.set_categories(labels)
chart_percent.title = "Part du CA pour chaque catégorie"

chart_percent.dataLabels = DataLabelList()
chart_percent.dataLabels.showPercent = True   # Afficher les pourcentages

# Cut the first slice out of the doughnut
slices = [DataPoint(idx=i) for i in range(4)]
plain, jam, lime, chocolate = slices
chart_percent.series[0].data_points = slices
plain.graphicalProperties.solidFill = "A9E6EB"
jam.graphicalProperties.solidFill = "B7A9EB"
lime.graphicalProperties.solidFill = "EBA9D7"

ws_visualisations.add_chart(chart_percent, "C27")


### La méthode ABC

Récupération des valeurs uniques des noms de produits

In [413]:
len_products = len(dataset["Product Name"].unique())+1
# Calcul du nombre de produits
len_dict ={}
for col in ["Product Name"]:
    len_dict["len_products"] = len(dataset[col].unique())+1 # on rajoute +1 car python commence à 0
len_dict

formula = "=_xlfn.UNIQUE(DATA!Q:Q)"
ws_data['Z33'] = ArrayFormula(f"Z33:Z{len_dict['len_products']+1}", formula)

Calcul du CA par produit

In [414]:
for num_cell in range(34, len_dict['len_products']+36):
    cell = "Y" + str(num_cell)
    ws_data[cell] = "=SUMIFS(DATA!R:R,DATA!Q:Q,DATA!Z" + str(num_cell) + ",DATA!M:M,Visualisations!$B$5, DATA!K:K, VISUALISATIONS!$B$11)"
# Calcul du total de CA
ws_data["Y33"] = f"=SUM(DATA!Y34:Y{len_dict['len_products']+1})"

Calcul part du CA du produit sur le Ca total

In [415]:
for num_cell in range(34, len_dict['len_products']+36):
    ws_data[f"X{num_cell}"] = f"=Y{num_cell}/$Y$33"

Tri dans l'ordre décroissant en fonction du niveau de la part

In [416]:
# Trier le tableau de données
formula = f"=_xlfn.SORT(DATA!X34:Z{len_dict['len_products']+36},1,-1)"
ws_data[f'Y{len_dict['len_products']+40}'] = ArrayFormula(f"Y{len_dict['len_products']+40}:AA{len_dict['len_products']*2+37}", formula)

Calcul de la part cumulée et classification en A, B ou C du produit

In [417]:
start_cell = len_dict['len_products'] + 40
end_cell = len_dict['len_products'] * 2 + 40

# Gérer le premier cas
ws_data[f"X{start_cell}"] = f"=Y{start_cell}"
ws_data[f"W{start_cell}"] = f'=IF(X{start_cell} <= 0.8, "A", IF(X{start_cell} <= 0.95, "B", "C"))'

# Boucle pour le reste des cellules
for num_cell in range(start_cell + 1, end_cell + 1):
    # Cumul
    ws_data[f"X{num_cell}"] = f"=Y{num_cell} + X{num_cell-1}"
    
    # Classification ABC
    ws_data[f"W{num_cell}"] = f'=IF(X{num_cell} <= 0.8, "A", IF(X{num_cell} <= 0.95, "B", "C"))'

Affichage dans la feuille visualisations

In [418]:
# On définit une plage de fin grande
row_fin = 7 + len_dict['len_products'] 

# Applications des formules
formula_a = f'=_xlfn.FILTER(DATA!AA{start_cell}:AA{end_cell}, DATA!W{start_cell}:W{end_cell}="A", "Aucun")'
formula_b = f'=_xlfn.FILTER(DATA!AA{start_cell}:AA{end_cell}, DATA!W{start_cell}:W{end_cell}="B", "Aucun")'
formula_c = f'=_xlfn.FILTER(DATA!AA{start_cell}:AA{end_cell}, DATA!W{start_cell}:W{end_cell}="C", "Aucun")'

ws_visualisations["K7"] = ArrayFormula(f"K7:k{row_fin}", formula_a)
ws_visualisations["N7"] = ArrayFormula(f"N7:N{row_fin}", formula_b)
ws_visualisations["P7"] = ArrayFormula(f"P7:P{row_fin}", formula_c)

# Largeur des colonnes
ws_visualisations.column_dimensions['K'].width = 45
ws_visualisations.column_dimensions['N'].width = 45
ws_visualisations.column_dimensions['P'].width = 45

alignement_propre = Alignment(wrap_text=True, vertical='top')

for col_lettre in ['K', 'N', 'P']:
    for row in range(7, row_fin + 1):
        ws_visualisations[f"{col_lettre}{row}"].alignment = alignement_propre

# Création de l'aspect design

### Le titre

In [419]:
ws_visualisations["K1"] = "Performances économique"
ws_visualisations["K1"].font = Font(bold=True, size=28, color="185EB8")

### KPIs numériques

In [420]:
def design_numerical_kpis(cell, nom):
    ws_visualisations[cell] = nom
    ws_visualisations[cell].font = Font(bold=True, size=16)

In [421]:
# CA
design_numerical_kpis("C3", "Chiffre d'affaires")

# Profits
design_numerical_kpis("E3", "Profits")

# Quantités
design_numerical_kpis("G3", "Volume des ventes")

# Temps livraison
design_numerical_kpis("J3", "Temps de livraison moyen (en jours)")


### Design Méthode ABC

In [422]:
def create_title_abc(cell_ref, letter, color):
    ws_visualisations[cell_ref] = letter
    ws_visualisations[cell_ref].font = Font(bold=True, size=14)
    ws_visualisations[cell_ref].fill = PatternFill(start_color=color, fill_type="solid")
    ws_visualisations[cell_ref].alignment = Alignment(horizontal="center", vertical="center")

In [423]:
create_title_abc("K6", "A", "56E871") # A
create_title_abc("N6", "B", "E8DC56") # B
create_title_abc("P6", "C", "E87B56") # C

### Design de la partie filtre

In [424]:
# largeur des colonnes
ws_visualisations.column_dimensions['A'].width = 16.5
ws_visualisations.column_dimensions['B'].width = 17.56

# Fusionner les cellules A4 et B4
ws_visualisations.merge_cells('A4:B4')

# Ajout du titre Filtres
ws_visualisations['A4'] = "Filtres"

# Appliquer le style gras et l'alignement
ws_visualisations['A4'].font = Font(bold=True, size=15)
ws_visualisations['A4'].alignment = Alignment(horizontal="center", vertical="center")

# création du fond et de la bordure
fond_gris = PatternFill(start_color="F2F2F2", fill_type="solid")
bordure_fine = Border(
    left=Side(border_style="thin", color="A6A6A6"),
    right=Side(border_style="thin", color="A6A6A6"),
    top=Side(border_style="thin", color="A6A6A6"),
    bottom=Side(border_style="thin", color="A6A6A6")
)

# Mettre un fond gris
for num_cell in range(4, 24):
    ws_visualisations[f'A{num_cell}'].fill = fond_gris
    ws_visualisations[f'B{num_cell}'].fill = fond_gris

# Mettre une bourdure
ws_visualisations['A4'].border = bordure_fine
ws_visualisations['B4'].border = bordure_fine

# Ajout du filtre

Création d'une fonction pour automatiser la mise en place d'un filtre. Lors de l'ajout d'un filtre, il faut également modifier les fonctions SUMIFS et AVERAGEIFS pour ajouter la référence de la cellule en condition.

In [425]:
def create_filter(column_ref, title, cell_ref_title, cell_ref_filter) :
    # Récupérer les valeurs uniques de la colonne à filtrer
    distinct_value_list = ",".join(dataset[column_ref].unique())
    ws_visualisations[cell_ref_title] = title
    ws_visualisations[cell_ref_filter] = dataset[column_ref][0]  # Valeur par défaut

    # liste de valeurs du filtre
    dv = DataValidation(
        type="list", formula1=f'"{distinct_value_list}"', allow_blank=True
    )

    # Ajout du filtre dans la feuille
    ws_visualisations.add_data_validation(dv)
    dv.add(ws_visualisations[cell_ref_filter])

Création des filtres

In [426]:
# Région
create_filter("Region", "Choisir une région :", "A5", "B5")

# États
create_filter("State", "Choisir un État :", "A11", "B11")

# Export du reporting

Sauvegarde du fichier

In [427]:
chemin_local = "../output/reporting.xlsx"
wb.save(chemin_local)

Export sur Minio

In [428]:
# Récupération de l'URL de MinIO
s3_endpoint = os.environ.get("AWS_S3_ENDPOINT")

# Initialisation de la connexion S3
s3 = boto3.client("s3", endpoint_url=f"https://{s3_endpoint}")

# Définition des chemins
nom_du_bucket = "oscar04" 
chemin_minio = "Superstore/reporting.xlsx" # Le dossier et le nom du fichier sur ton espace cloud

# Envoi du fichier
s3.upload_file(chemin_local, nom_du_bucket, chemin_minio)

print(f"✅ Fichier exporté avec succès sur MinIO dans le bucket {nom_du_bucket}/{chemin_minio}")

✅ Fichier exporté avec succès sur MinIO dans le bucket oscar04/Superstore/reporting.xlsx
